# Nano Video Generation: All-in-One Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshkanTaghipour/nano-video-gen/blob/main/notebooks/nano_video_gen_colab.ipynb)

A self-contained notebook that teaches the concepts behind modern video diffusion models (Wan 2.1 architecture) with a ~2.0M parameter **NanoDiT** model you can train and run inference on in minutes.

The NanoDiT uses the **real pretrained Wan 2.1 VAE** and **T5 text encoder** -- the same components used by the full-scale models -- so it operates on real latent representations and real text embeddings.

**What you will learn:**
1. How the VAE compresses video into a latent space
2. Every building block of the Diffusion Transformer (RoPE, RMSNorm, AdaIN, attention, FFN)
3. How the full DiT architecture assembles these blocks
4. The flow matching diffusion paradigm (training & inference)
5. Training a tiny video generation model on synthetic data with real VAE + T5
6. Generating videos with Classifier-Free Guidance (CFG)

**Runtime**: ~20 minutes end-to-end on a Colab T4 GPU (includes ~9.5 GB model download on first run).

---
## Setup

Clone the repository and install dependencies. This cell only needs to run once per Colab session.

In [ ]:
import os

# Clone the repo (skip if already cloned)
if not os.path.exists('/content/nano-video-gen'):
    !git clone https://github.com/AshkanTaghipour/nano-video-gen.git /content/nano-video-gen

%cd /content/nano-video-gen

# Install dependencies (no conda on Colab)
!pip install -q torch torchvision einops matplotlib imageio imageio-ffmpeg Pillow tqdm \
    diffusers transformers accelerate sentencepiece huggingface_hub

In [ ]:
import sys
import torch

# Ensure the project root is on the Python path
sys.path.insert(0, '/content/nano-video-gen')

# Verify GPU
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    device = torch.device('cuda')
else:
    print("No GPU detected -- running on CPU (training will be slower).")
    device = torch.device('cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

---
# Section 1: Video Generation Overview

Video generation synthesizes realistic video frames from a conditioning signal -- typically a text prompt. Adding the **temporal dimension** to image generation introduces major challenges:

| Aspect | Image Generation | Video Generation |
|--------|-----------------|------------------|
| Dimensionality | 2D (H x W) | 3D (T x H x W) |
| Consistency | Single frame | Must be coherent across many frames |
| Compute | Manageable | Grows linearly (or worse) with frame count |
| Motion | N/A | Must model physically plausible motion |

Modern video diffusion models like **Wan 2.1** tackle these challenges with three core components:

1. **Video VAE** -- compresses raw pixel-space video into a compact *latent* representation
2. **Text Encoder** -- converts a natural-language prompt into dense embeddings
3. **Diffusion Transformer (DiT)** -- iteratively *denoises* noise in latent space, guided by text

### Pipeline Flow

```
Text Prompt
    |  
    v
[Text Encoder] --> text embeddings (B, seq_len, text_dim)
    |  
    v
Random Noise z_T ~ N(0, I)  in latent space (B, C_lat, T', H', W')
    |  
    v  (iterative denoising, N steps)
[DiT Model]  x  text embeddings  x  timestep  -->  predicted velocity
    |  
    v  (Euler step: z_{t-1} = z_t + v * delta_sigma)
Clean latent z_0  (B, C_lat, T', H', W')
    |  
    v
[VAE Decoder]  -->  video pixels  (B, 3, T, H, W)
```

In [ ]:
# Architecture comparison: Wan 14B vs Wan 1.3B vs Nano
#
# Our Nano model preserves every architectural concept
# from the full Wan model, just at a much smaller scale.
# It uses the SAME pretrained VAE and T5 text encoder.

header = f"{'Component':<20} {'Wan 14B':>10} {'Wan 1.3B':>10} {'Nano':>10}"
sep    = '-' * len(header)

rows = [
    ('DiT dim',           '5120',  '1536',  '128'),
    ('Attention heads',   '40',    '12',    '4'),
    ('Transformer layers','40',    '30',    '2'),
    ('FFN dim',           '13824', '8960',  '512'),
    ('Latent channels',   '16',    '16',    '16'),
    ('Text dim',          '4096',  '4096',  '4096'),
    ('Freq dim',          '256',   '256',   '64'),
    ('Patch size',        '[1,2,2]','[1,2,2]','[1,2,2]'),
    ('~Parameters',       '~14B',  '~1.3B', '~2.0M'),
]

print(header)
print(sep)
for name, wan14, wan1, nano in rows:
    print(f"{name:<20} {wan14:>10} {wan1:>10} {nano:>10}")

print()
print("Note: Nano uses the SAME pretrained Wan 2.1 VAE (16 latent ch)")
print("and T5 text encoder (4096-dim) -- only the DiT is tiny.")

In [ ]:
# Compute the compression ratio of latent diffusion
# Using the real Wan 2.1 VAE dimensions

# Pixel-space video dimensions (128x128, 17 frames)
B, C, T, H, W = 1, 3, 17, 128, 128
pixel_values = C * T * H * W

# Latent-space dimensions after Wan VAE encoding
# Spatial: 8x compression, Temporal: (T-1)//4 + 1
C_lat = 16
T_lat = (T - 1) // 4 + 1  # 5
H_lat, W_lat = H // 8, W // 8  # 16, 16
latent_values = C_lat * T_lat * H_lat * W_lat

ratio = pixel_values / latent_values

print(f"Pixel space:  [{B}, {C}, {T}, {H}, {W}] = {pixel_values:,} values per sample")
print(f"Latent space: [{B}, {C_lat}, {T_lat}, {H_lat}, {W_lat}] = {latent_values:,} values per sample")
print(f"Compression ratio: {ratio:.0f}x")
print()
print("The DiT model only needs to denoise ~20K values")
print("instead of ~786K values per diffusion step!")

---
# Section 2: VAE & Latent Space

The **Video VAE** compresses raw pixel-space video into a compact latent representation suitable for diffusion.

- **Encoder**: maps `[B, 3, T, H, W]` to a low-dimensional latent `[B, C_lat, T', H', W']`
- **Decoder**: reconstructs the video from the latent back to `[B, 3, T, H, W]`

Here we use a `DummyVAE` to illustrate the encode/decode concept with simple Conv3d layers. In sections 6-7, we switch to the **real pretrained Wan 2.1 VAE** (CausalConv3d, 16 latent channels, 8x spatial + 4x temporal compression).

In [ ]:
import matplotlib.pyplot as plt
from nano_video_gen.model.nano_vae import DummyVAE
from nano_video_gen.visualization import visualize_latent_space

# Instantiate the DummyVAE
vae = DummyVAE(
    in_channels=3,
    latent_channels=4,
    spatial_factor=4,
    temporal_factor=4,
)

total_params = sum(p.numel() for p in vae.parameters())
print(f"DummyVAE parameters: {total_params:,}")
print(vae)

In [ ]:
# Encode -> Decode demo
torch.manual_seed(42)
video = torch.randn(1, 3, 16, 64, 64).clamp(-1, 1)
print(f"Input video shape:          {list(video.shape)}")
print(f"Input values per sample:    {video[0].numel():,}")

vae.eval()
with torch.no_grad():
    latent = vae.encode(video)
    reconstruction = vae.decode(latent)

print(f"Latent shape:               {list(latent.shape)}")
print(f"Compression ratio:          {video[0].numel() / latent[0].numel():.1f}x")
print(f"Reconstruction shape:       {list(reconstruction.shape)}")

In [ ]:
# Visualize latent channels
fig = visualize_latent_space(
    latent,
    title="DummyVAE Latent Channels (first temporal frame)"
)
plt.show()

In [ ]:
# Reconstruction error
with torch.no_grad():
    mse = torch.nn.functional.mse_loss(reconstruction, video)
    mae = torch.nn.functional.l1_loss(reconstruction, video)
    psnr = 10 * torch.log10(4.0 / mse)

print("=== Reconstruction Error (untrained DummyVAE) ===")
print(f"MSE:  {mse.item():.4f}")
print(f"MAE:  {mae.item():.4f}")
print(f"PSNR: {psnr.item():.2f} dB")
print()
print("An untrained VAE has high error. A trained VAE achieves PSNR > 30 dB.")

# Visualize original vs reconstruction
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

orig_frame = ((video[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()
recon_frame = ((reconstruction[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()

axes[0].imshow(orig_frame)
axes[0].set_title('Original (frame 0)')
axes[0].axis('off')

axes[1].imshow(recon_frame)
axes[1].set_title(f'Reconstruction (PSNR={psnr.item():.1f} dB)')
axes[1].axis('off')

plt.suptitle('VAE Encode -> Decode', fontsize=14)
plt.tight_layout()
plt.show()

---
# Section 3: Building Blocks of the DiT

The Diffusion Transformer is built from a small set of reusable components. Let's examine each one.

## 3.1 Sinusoidal Embeddings

The diffusion model encodes the scalar timestep $t$ into a high-dimensional vector using sine/cosine functions at exponentially spaced frequencies. Low frequencies distinguish early vs. late timesteps; high frequencies distinguish nearby timesteps.

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

from nano_video_gen.model.components import (
    sinusoidal_embedding_1d,
    precompute_freqs_cis_3d,
    rope_apply,
    RMSNorm,
    modulate,
    GateModule,
)
from nano_video_gen.model.attention import SelfAttention, CrossAttention
from nano_video_gen.visualization import (
    visualize_rope_frequencies,
    visualize_modulation_effect,
    visualize_attention_maps,
    visualize_ffn_activations,
)

torch.manual_seed(42)

# Sinusoidal embeddings for various timesteps
dim = 64
timesteps = torch.tensor([0, 50, 100, 250, 500, 750, 1000], dtype=torch.float32)
embeddings = sinusoidal_embedding_1d(dim, timesteps)

print(f"Timesteps: {timesteps.tolist()}")
print(f"Embedding shape: {list(embeddings.shape)}  (num_timesteps x dim)")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

im = axes[0].imshow(embeddings.numpy(), cmap='RdBu', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Timestep')
axes[0].set_yticks(range(len(timesteps)))
axes[0].set_yticklabels([f't={int(t)}' for t in timesteps])
axes[0].set_title('Sinusoidal Timestep Embeddings')
plt.colorbar(im, ax=axes[0])

dense_t = torch.linspace(0, 1000, 200)
dense_emb = sinusoidal_embedding_1d(dim, dense_t)
for ch in [0, 8, 16, 31]:
    axes[1].plot(dense_t.numpy(), dense_emb[:, ch].numpy(), label=f'dim {ch}')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Embedding value')
axes[1].set_title('Individual frequency channels (low to high frequency)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3.2 3D Rotary Position Embeddings (RoPE)

Video tokens live in a 3D grid: **(T, H, W)**. RoPE encodes position by *rotating* pairs of dimensions in the query/key vectors. The dot product $q \cdot k$ naturally encodes **relative** position.

For 3D video, the head dimension is split into temporal, height, and width frequency parts.

In [ ]:
head_dim = 32  # dim=128, num_heads=4 -> head_dim=32

freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)

print(f"Head dimension: {head_dim}")
print(f"Split: temporal={head_dim - 2*(head_dim//3)}, height={head_dim//3}, width={head_dim//3}")
print(f"Temporal freqs shape: {list(freqs_t.shape)}")
print(f"Height freqs shape:   {list(freqs_h.shape)}")
print(f"Width freqs shape:    {list(freqs_w.shape)}")

fig = visualize_rope_frequencies((freqs_t, freqs_h, freqs_w), max_pos=32)
plt.show()

## 3.3 RMSNorm vs LayerNorm

**RMSNorm** is a simplified LayerNorm used throughout the DiT. It normalizes by the root mean square (no mean subtraction), making it ~5-10% faster.

In [ ]:
dim = 128
rms_norm = RMSNorm(dim)
layer_norm = nn.LayerNorm(dim)

torch.manual_seed(42)
x = torch.randn(4, 16, dim) * 3.0 + 2.0

with torch.no_grad():
    x_rms = rms_norm(x)
    x_ln = layer_norm(x)

print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"RMSNorm -- mean: {x_rms.mean():.4f}, std: {x_rms.std():.4f}")
print(f"LayerNorm -- mean: {x_ln.mean():.4f}, std: {x_ln.std():.4f}")
print()
print("RMSNorm does NOT center the data (mean != 0).")
print("LayerNorm centers (mean ~ 0) AND normalizes variance.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(x.flatten().numpy(), bins=80, alpha=0.7, color='gray', edgecolor='black')
axes[0].set_title(f'Input (mean={x.mean():.2f}, std={x.std():.2f})')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[1].hist(x_rms.flatten().numpy(), bins=80, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].set_title(f'RMSNorm (mean={x_rms.mean():.2f}, std={x_rms.std():.2f})')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[2].hist(x_ln.flatten().numpy(), bins=80, alpha=0.7, color='coral', edgecolor='black')
axes[2].set_title(f'LayerNorm (mean={x_ln.mean():.2f}, std={x_ln.std():.2f})')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('RMSNorm vs LayerNorm: Output Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 3.4 AdaIN Modulation

**Adaptive Instance Normalization** modulation is how the timestep controls each layer:

$$\text{output} = x \cdot (1 + \text{scale}) + \text{shift}$$

Each DiT block has **6 modulation parameters**: shift/scale/gate for self-attention, and shift/scale/gate for the FFN.

In [ ]:
dim = 128
torch.manual_seed(42)
x = torch.randn(1, 16, dim)
shift = torch.randn(1, 1, dim) * 0.5
scale = torch.randn(1, 1, dim) * 0.3

x_modulated = modulate(x, shift, scale)

print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"Output -- mean: {x_modulated.mean():.4f}, std: {x_modulated.std():.4f}")

fig = visualize_modulation_effect(x, x_modulated, shift, scale)
plt.show()

## 3.5 Self-Attention

**Self-attention** allows each video patch to attend to every other patch. In the Wan architecture:
1. Input $x$ is projected to $Q$, $K$, $V$
2. $Q$ and $K$ are **RMSNorm'd** (stabilizes training)
3. **3D RoPE** is applied to $Q$ and $K$
4. Scaled dot-product attention: $\text{Attn} = \text{softmax}(QK^T / \sqrt{d_k}) V$

In [ ]:
dim = 128
num_heads = 4
head_dim = dim // num_heads

self_attn = SelfAttention(dim=dim, num_heads=num_heads)
self_attn.eval()

torch.manual_seed(42)
f, h, w = 2, 4, 4
seq_len = f * h * w  # 32 tokens
x = torch.randn(1, seq_len, dim)

# Build 3D RoPE frequencies
freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)
freqs = torch.cat([
    freqs_t[:f].view(f, 1, 1, -1).expand(f, h, w, -1),
    freqs_h[:h].view(1, h, 1, -1).expand(f, h, w, -1),
    freqs_w[:w].view(1, 1, w, -1).expand(f, h, w, -1)
], dim=-1).reshape(seq_len, 1, -1)

with torch.no_grad():
    output = self_attn(x, freqs)
    # Compute attention weights for visualization
    q = self_attn.norm_q(self_attn.q(x))
    k = self_attn.norm_k(self_attn.k(x))
    q = rope_apply(q, freqs, num_heads)
    k = rope_apply(k, freqs, num_heads)
    q = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
    k = rearrange(k, "b s (n d) -> b n s d", n=num_heads)
    attn_weights = F.softmax(torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5), dim=-1)

print(f"Input: {list(x.shape)} -> Output: {list(output.shape)}")
print(f"Attention weights: {list(attn_weights.shape)}")

fig = visualize_attention_maps(
    attn_weights,
    title=f"Self-Attention Maps ({seq_len} tokens = {f}f x {h}h x {w}w)",
    num_heads_to_show=4,
)
plt.show()

## 3.6 Cross-Attention

**Cross-attention** is how text prompts guide video generation. Queries come from video features; keys and values come from text embeddings. No RoPE is applied (text tokens have their own encoding).

In [ ]:
cross_attn = CrossAttention(dim=dim, num_heads=num_heads)
cross_attn.eval()

torch.manual_seed(42)
x_video = torch.randn(1, seq_len, dim)
seq_len_text = 8
context = torch.randn(1, seq_len_text, dim)

with torch.no_grad():
    output = cross_attn(x_video, context)
    q = cross_attn.norm_q(cross_attn.q(x_video))
    k = cross_attn.norm_k(cross_attn.k(context))
    q = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
    k = rearrange(k, "b s (n d) -> b n s d", n=num_heads)
    cross_attn_weights = F.softmax(torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5), dim=-1)

print(f"Video input: {list(x_video.shape)} -> Output: {list(output.shape)}")
print(f"Cross-attention: {seq_len} video tokens attend to {seq_len_text} text tokens")

fig = visualize_attention_maps(
    cross_attn_weights,
    title=f"Cross-Attention Maps ({seq_len} video -> {seq_len_text} text tokens)",
    num_heads_to_show=4,
)
plt.show()

## 3.7 Feed-Forward Network (FFN)

The FFN provides per-token non-linear capacity: $\text{FFN}(x) = \text{Linear}_2(\text{GELU}(\text{Linear}_1(x)))$

Nano uses dim=128 -> ffn_dim=512 (4x expansion).

In [ ]:
ffn_dim = 512
ffn = nn.Sequential(
    nn.Linear(dim, ffn_dim),
    nn.GELU(approximate='tanh'),
    nn.Linear(ffn_dim, dim)
)

torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)

with torch.no_grad():
    h = ffn[0](x)
    h_gelu = ffn[1](h)
    output = ffn[2](h_gelu)

print(f"Input: {list(x.shape)} -> After Linear1: {list(h.shape)} -> After GELU: {list(h_gelu.shape)} -> Output: {list(output.shape)}")

fig = visualize_ffn_activations(h_gelu, title="FFN Hidden Activations (after GELU)")
plt.show()

## 3.8 Gate Module

The **GateModule** provides learned gating for residual connections:
$\text{output} = x + \text{gate} \cdot f(x)$

The gate is predicted from the timestep embedding, allowing the model to control how much each layer contributes at each noise level.

In [ ]:
gate_module = GateModule()

torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)
residual = torch.randn(1, seq_len, dim)

gate_values = [0.0, 0.5, 1.0, 2.0, -0.5]
fig, axes = plt.subplots(1, len(gate_values), figsize=(16, 3))

for i, g in enumerate(gate_values):
    gate = torch.full((1, 1, dim), g)
    output = gate_module(x, gate, residual)
    diff = (output - x).flatten().detach().numpy()
    axes[i].hist(diff, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[i].set_title(f'gate = {g}')
    axes[i].set_xlabel('output - x')
    axes[i].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    axes[i].set_xlim(-6, 6)

axes[0].set_ylabel('Count')
plt.suptitle('GateModule: output = x + gate * residual', fontsize=14)
plt.tight_layout()
plt.show()

print("gate=0.0: No residual (output = x)")
print("gate=1.0: Standard residual (output = x + residual)")
print("gate=2.0: Amplified residual")
print("gate=-0.5: Subtractive gating")

---
# Section 4: DiT Architecture

The **NanoDiT** assembles all building blocks into a complete Diffusion Transformer. Key differences from a standard LLM Transformer:

| Aspect | LLM Transformer | DiT |
|--------|-----------------|-----|
| Input | Token embeddings | Patchified noisy latents |
| Self-attention | Causal (masked) | Bidirectional (no mask) |
| Conditioning | None / prefix | Timestep modulation via AdaIN |
| Cross-attention | None | Text conditioning |
| Positional encoding | 1D | 3D RoPE |
| Output | Next-token logits | Predicted velocity $v = \epsilon - x_0$ |

In [ ]:
from nano_video_gen.model.nano_dit import NanoDiT

# For this architectural demo we use small dims to keep it fast.
# In sections 6-7 we use the real dims: in_dim=16, text_dim=4096.
model = NanoDiT(
    dim=128, in_dim=4, ffn_dim=512, out_dim=4,
    text_dim=64, freq_dim=64, num_heads=4, num_layers=2,
    patch_size=(1, 2, 2),
)
model.eval()

print("=" * 70)
print("NanoDiT Model Architecture (demo dims for illustration)")
print("=" * 70)
print(model)

In [ ]:
# Parameter breakdown
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

components = {
    "patch_embedding": model.patch_embedding,
    "text_embedding":  model.text_embedding,
    "time_embedding":  model.time_embedding,
    "time_projection": model.time_projection,
}
for i, block in enumerate(model.blocks):
    components[f"blocks[{i}]"] = block
components["head"] = model.head

total = count_params(model)
print(f"{'Component':<22} {'Parameters':>12}  {'% of Total':>10}")
print("-" * 48)
for name, mod in components.items():
    n = count_params(mod)
    print(f"{name:<22} {n:>12,}  {n / total * 100:>9.1f}%")
print("-" * 48)
print(f"{'TOTAL':<22} {total:>12,}  100.0%")

In [ ]:
# Forward pass
B = 1
x = torch.randn(B, 4, 4, 16, 16)       # [B, in_dim, T, H, W]
timestep = torch.tensor([500.0])         # [B]
context = torch.randn(B, 8, 64)          # [B, seq_len, text_dim]

print(f"Input:    x={list(x.shape)}, timestep={timestep.item():.0f}, context={list(context.shape)}")

with torch.no_grad():
    output = model(x, timestep, context)

print(f"Output:   {list(output.shape)}")
print(f"Same dims as input: {list(x.shape) == list(output.shape)}")

In [ ]:
# Data flow visualization
from nano_video_gen.visualization.viz import visualize_data_flow

fig = visualize_data_flow(model, x, timestep, context, figsize=(16, 5))
plt.show()

In [ ]:
import math

def estimate_params(
    dim, ffn_dim, num_layers, num_heads,
    in_dim, out_dim, text_dim, freq_dim,
    patch_size=(1, 2, 2),
):
    patch_vol = math.prod(patch_size)
    p_patch = in_dim * dim * patch_vol + dim
    p_text = (text_dim * dim + dim) + (dim * dim + dim)
    p_time = (freq_dim * dim + dim) + (dim * dim + dim)
    p_time_proj = dim * (dim * 6) + (dim * 6)
    p_self_attn = 4 * (dim * dim + dim) + 2 * dim
    p_cross_attn = 4 * (dim * dim + dim) + 2 * dim
    p_ffn = (dim * ffn_dim + ffn_dim) + (ffn_dim * dim + dim)
    p_norms = dim + dim
    p_mod = 6 * dim
    p_per_block = p_self_attn + p_cross_attn + p_ffn + p_norms + p_mod
    p_blocks = p_per_block * num_layers
    p_head = (dim * (out_dim * patch_vol) + out_dim * patch_vol) + 2 * dim
    total = p_patch + p_text + p_time + p_time_proj + p_blocks + p_head
    return {f"blocks (x{num_layers})": p_blocks, "TOTAL": total}

configs = {
    "Nano (~1.5M)": dict(dim=128, ffn_dim=512, num_layers=2, num_heads=4, in_dim=4, out_dim=4, text_dim=64, freq_dim=64),
    "Wan 1.3B": dict(dim=1536, ffn_dim=8960, num_layers=30, num_heads=12, in_dim=16, out_dim=16, text_dim=4096, freq_dim=256),
    "Wan 14B": dict(dim=5120, ffn_dim=13824, num_layers=40, num_heads=40, in_dim=16, out_dim=16, text_dim=4096, freq_dim=256),
}

results = {name: estimate_params(**cfg) for name, cfg in configs.items()}
nano_total = results["Nano (~1.5M)"]["TOTAL"]

print("--- Scaling Comparison ---")
for name in configs:
    t = results[name]["TOTAL"]
    print(f"  {name:>14}: {t:>14,} params  ({t / nano_total:,.0f}x vs Nano)")

print(f"\nKey insight: dim 128 -> 5120 is {5120//128}x width. Since params scale as O(dim^2),")
print(f"this alone gives ~{(5120/128)**2:.0f}x more params. Combined with 40 layers (vs 2) = ~{results['Wan 14B']['TOTAL'] / nano_total:,.0f}x total.")

---
# Section 5: Flow Matching

**Flow matching** is the diffusion paradigm used by Wan 2.1 (and FLUX, SD3). It is simpler and faster than DDPM.

| Aspect | DDPM | Flow Matching |
|--------|------|---------------|
| Forward process | $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$ | $x_t = (1-t) x_0 + t \epsilon$ |
| Training target | Noise $\epsilon$ | Velocity $v = \epsilon - x_0$ |
| Denoising step | Complex variance formula | Simple Euler step |
| Convergence | 1000s of steps | 20-50 steps sufficient |

In [ ]:
from nano_video_gen.diffusion.flow_match import FlowMatchScheduler

# Compare sigma schedules for different shift values
shifts = [1, 3, 5]
num_steps = 50

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2196F3', '#FF9800', '#E91E63']

for shift, color in zip(shifts, colors):
    scheduler = FlowMatchScheduler()
    scheduler.set_timesteps(num_inference_steps=num_steps, shift=shift)
    sigmas = scheduler.sigmas.numpy()
    timesteps_arr = scheduler.timesteps.numpy()
    step_indices = np.arange(len(sigmas))

    axes[0].plot(step_indices, sigmas, '-o', color=color, markersize=3, label=f'shift={shift}', linewidth=2)
    axes[1].plot(step_indices, timesteps_arr, '-o', color=color, markersize=3, label=f'shift={shift}', linewidth=2)

axes[0].set_title('Sigma (noise level) vs Step Index', fontsize=13)
axes[0].set_xlabel('Denoising step')
axes[0].set_ylabel('Sigma')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.02, 1.05)

axes[1].set_title('Timestep vs Step Index', fontsize=13)
axes[1].set_xlabel('Denoising step')
axes[1].set_ylabel('Timestep (0-1000)')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Effect of Shift on the Noise Schedule', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("With shift=5 (Wan default), ~60% of steps are at sigma > 0.5,")
print("giving more compute for the hardest (high-noise) denoising steps.")

In [ ]:
# Visualize the forward noising process
H, W = 64, 64
y_grid, x_grid = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
checker = ((torch.floor(x_grid * 4) + torch.floor(y_grid * 4)) % 2).float()
circle = 1.0 - (x_grid ** 2 + y_grid ** 2).sqrt().clamp(0, 1)
x0 = torch.stack([checker, circle, 0.5 * (checker + circle)], dim=0) * 2 - 1

torch.manual_seed(42)
noise = torch.randn_like(x0)

t_values = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(t_values), figsize=(16, 3.5))

for ax, t in zip(axes, t_values):
    x_t = (1 - t) * x0 + t * noise
    img = ((x_t + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f'$t = {t:.2f}$', fontsize=12)
    ax.axis('off')

fig.suptitle('Forward Process: $x_t = (1-t) \\cdot x_0 + t \\cdot \\epsilon$', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Training loss on random data (demonstrates the mechanics)
torch.manual_seed(0)
model_fm = NanoDiT()
model_fm.eval()
scheduler = FlowMatchScheduler()

x0 = torch.randn(1, 4, 4, 16, 16)
context = torch.randn(1, 8, 64)
loss = scheduler.compute_loss(model_fm, x0, context)

print(f"Flow matching loss (random init): {loss.item():.4f}")
print(f"Expected for random predictor: ~2.0 (since Var(noise - x0) = 2)")
print("With training, this decreases as the model learns the velocity field.")

In [ ]:
# Demonstrate a single Euler denoising step
torch.manual_seed(123)
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(num_inference_steps=10, shift=5.0)

print("Sigma schedule (10 steps, shift=5):")
print(f"  Sigmas:    {scheduler.sigmas.numpy().round(4)}")
print(f"  Timesteps: {scheduler.timesteps.numpy().round(1)}")

x_t = torch.randn(1, 4, 4, 16, 16)
v_pred = torch.randn_like(x_t) * 0.1

sigma_current = scheduler.sigmas[0].item()
sigma_next = scheduler.sigmas[1].item()
delta_sigma = sigma_next - sigma_current

x_t_minus_1 = x_t + v_pred * delta_sigma

print(f"\nEuler step: sigma {sigma_current:.4f} -> {sigma_next:.4f} (delta={delta_sigma:.4f})")
print(f"x_t stats:     mean={x_t.mean().item():.4f}, std={x_t.std().item():.4f}")
print(f"x_{{t-1}} stats: mean={x_t_minus_1.mean().item():.4f}, std={x_t_minus_1.std().item():.4f}")

# Verify against scheduler.step()
x_check = scheduler.step(v_pred, scheduler.timesteps[0], x_t)
print(f"\nManual vs scheduler.step() max diff: {(x_t_minus_1 - x_check).abs().max().item():.2e}")

---
# Section 6: Training

Now we train our NanoDiT on a synthetic moving-shapes dataset using the **pretrained Wan 2.1 VAE** (16 latent channels, CausalConv3d) and **umt5-xxl T5 text encoder** (4096-dim embeddings).

The pipeline:
1. Download pretrained VAE and T5 weights (~9.5 GB, cached after first run)
2. Generate synthetic dataset at 128x128, 17 frames
3. Pre-compute T5 text embeddings for all prompts, then free the T5 model (~9 GB)
4. Train NanoDiT with `in_dim=16, out_dim=16, text_dim=4096` (~2.0M params)

Only the NanoDiT trains -- the VAE and T5 are pretrained and frozen. The VAE runs on CPU to save GPU VRAM.

In [ ]:
import os
import gc
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from nano_video_gen.model.nano_dit import NanoDiT
from nano_video_gen.model.wan_vae_wrapper import WanVAEWrapper
from nano_video_gen.model.t5_text_encoder import T5TextEncoder, CachedTextEmbeddings
from nano_video_gen.diffusion.flow_match import FlowMatchScheduler
from nano_video_gen.data.dataset import VideoDataset
from nano_video_gen.visualization.viz import plot_training_curves, save_video_grid

### 6.1 Download Pretrained Models & Generate Synthetic Dataset

We download the Wan 2.1 VAE and T5 text encoder (~9.5 GB total) and generate synthetic data at 128x128 resolution with 17 frames.

In [ ]:
model_path = '/content/nano-video-gen/pretrained_models/Wan2.1'

# Generate synthetic dataset at 128x128 with 17 frames
data_dir = '/content/nano-video-gen/data/synthetic_dataset'

if not os.path.exists(data_dir):
    print("Generating synthetic dataset (128x128, 17 frames)...")
    from nano_video_gen.data.generate_synthetic import generate_dataset
    generate_dataset(output_dir=data_dir, height=128, width=128, num_frames=17)
    print("Done!")
else:
    print(f"Dataset already exists at: {data_dir}")

# Load WanVAE on CPU (downloads weights if needed)
print("\nLoading Wan 2.1 VAE on CPU...")
vae = WanVAEWrapper(model_path=model_path, device="cpu")
print(f"WanVAE loaded (latent_channels={vae.latent_channels})")

# Quick test: verify latent shape
with torch.no_grad():
    test_video = torch.randn(1, 3, 17, 128, 128)
    test_latent = vae.encode(test_video)
    latent_shape = (1,) + tuple(test_latent.shape[1:])
    print(f"Test encode: [1, 3, 17, 128, 128] -> {list(test_latent.shape)}")
    del test_video, test_latent

In [ ]:
# Create dataset and dataloader (128x128, 17 frames)
dataset = VideoDataset(
    base_path=data_dir,
    height=128,
    width=128,
    num_frames=17,
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

print(f"Dataset size: {len(dataset)} videos")
print(f"Unique prompts: {dataset.num_prompts}")
print(f"Batches per epoch: {len(dataloader)}")

# Collect unique prompts for T5 encoding
unique_prompts = sorted(set(dataset.prompts))
prompt_to_idx = {p: i for i, p in enumerate(unique_prompts)}
print(f"\nUnique prompts ({len(unique_prompts)}):")
for i, p in enumerate(unique_prompts):
    print(f"  [{i}] {p}")

# Pre-compute T5 text embeddings, then free T5 model (~9 GB)
print("\nLoading T5 text encoder...")
t5 = T5TextEncoder(model_path=model_path, device="cpu")

print("Encoding all prompts...")
all_embeddings = t5.encode(unique_prompts)
print(f"Text embeddings shape: {list(all_embeddings.shape)}")

t5.free_memory()
gc.collect()

# Create cached embeddings on GPU
cached_text = CachedTextEmbeddings(all_embeddings).to(device)
print(f"Cached text embeddings on {device}")

In [ ]:
# Initialize NanoDiT with real model dimensions
model = NanoDiT(
    dim=128, in_dim=16, ffn_dim=512, out_dim=16,
    text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
    patch_size=(1, 2, 2),
).to(device)

total_dit_params = sum(p.numel() for p in model.parameters())
print(f"NanoDiT:  {total_dit_params:>10,} params (~{total_dit_params/1e6:.1f}M)")
print(f"VAE:      pretrained (frozen, on CPU)")
print(f"Text:     cached T5 embeddings ({cached_text.num_prompts} prompts x {cached_text.seq_len} tokens x {cached_text.text_dim})")
print(f"Latent:   {list(latent_shape)}")
print(f"Device:   {device}")

# Compare with educational mode
print(f"\nVs educational mode: in_dim 4->16, text_dim 64->4096")
print(f"  Text projection adds ~{128*4096*2/1e6:.1f}M params (Linear(4096->128) + Linear(128->128))")

### 6.2 Training Loop

Only NanoDiT parameters are trained. The VAE encodes videos on CPU, and T5 embeddings are looked up from the cache.

In [ ]:
# Training configuration
num_epochs = 50 if torch.cuda.is_available() else 15
learning_rate = 1e-3
weight_decay = 0.01
max_grad_norm = 1.0

scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

# Only NanoDiT parameters are trainable
trainable_params = list(model.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

model.train()
vae.eval()

losses = []
epoch_losses = []

output_dir = '/content/nano-video-gen/outputs'
os.makedirs(output_dir, exist_ok=True)

print(f"Training for {num_epochs} epochs...")
print(f"  Trainable params: {sum(p.numel() for p in trainable_params):,}")
print(f"  VAE: frozen on CPU | Text: cached T5 embeddings on {device}")

for epoch in range(num_epochs):
    epoch_loss_sum = 0.0
    epoch_steps = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch in progress_bar:
        video = batch["video"].to(device)
        prompt_idx = batch["prompt_idx"].to(device)

        # Look up cached T5 embeddings
        context = cached_text(prompt_idx)

        # Compute flow matching loss (VAE encodes on CPU inside compute_loss)
        loss = scheduler.compute_loss(model, video, context, vae=vae)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
        optimizer.step()

        loss_val = loss.item()
        losses.append(loss_val)
        epoch_loss_sum += loss_val
        epoch_steps += 1
        progress_bar.set_postfix(loss=f"{loss_val:.4f}")

    avg_epoch_loss = epoch_loss_sum / max(epoch_steps, 1)
    epoch_losses.append(avg_epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs}: avg_loss={avg_epoch_loss:.4f}")

print(f"\nTraining complete! Final loss: {losses[-1]:.4f}")

In [ ]:
# Loss curve
fig = plot_training_curves(losses, title="Flow Matching Training Loss")
plt.show()

In [ ]:
# Save checkpoint
final_ckpt_path = os.path.join(output_dir, 'checkpoint_final.pt')

torch.save({
    'model': model.state_dict(),
    'text_embeddings': cached_text.embeddings.cpu(),
    'epoch': num_epochs,
    'losses': losses,
    'num_prompts': cached_text.num_prompts,
    'latent_shape': list(latent_shape),
}, final_ckpt_path)

total_dit_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint saved to: {final_ckpt_path}")
print(f"  NanoDiT params: {total_dit_params:,}")
print(f"  Text embeddings: {list(cached_text.embeddings.shape)}")
print(f"  Latent shape: {list(latent_shape)}")
print(f"  Final loss:   {losses[-1]:.4f}")

---
# Section 7: Inference

Now we use the trained model to generate videos, decoding latents through the Wan 2.1 VAE. We will explore:
1. The full denoising loop with intermediate visualization
2. Classifier-Free Guidance (CFG)
3. Effect of denoising step count
4. Saving generated videos as GIF

In [ ]:
from nano_video_gen.visualization.viz import visualize_denoising_process

# Load checkpoint (in case this section is run independently)
checkpoint_path = '/content/nano-video-gen/outputs/checkpoint_final.pt'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Real models checkpoint
    latent_shape = tuple(checkpoint['latent_shape'])
    num_prompts = checkpoint['num_prompts']

    model = NanoDiT(
        dim=128, in_dim=16, ffn_dim=512, out_dim=16,
        text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
        patch_size=(1, 2, 2),
    ).to(device)
    model.load_state_dict(checkpoint['model'])

    cached_text = CachedTextEmbeddings(checkpoint['text_embeddings']).to(device)

    # Re-load WanVAE if not already in memory
    if 'vae' not in dir() or not isinstance(vae, WanVAEWrapper):
        model_path = '/content/nano-video-gen/pretrained_models/Wan2.1'
        vae = WanVAEWrapper(model_path=model_path, device="cpu")

    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"Latent shape: {list(latent_shape)}")
else:
    print("Using model from training (already in memory).")

model.eval()
vae.eval()
cached_text.eval()
print("Models set to eval mode.")

### 7.1 Denoising Loop with Visualization

In [ ]:
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

x = torch.randn(latent_shape, device=device)
ctx = cached_text(torch.tensor([0], device=device))

denoising_frames = []

with torch.no_grad():
    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Denoising (50 steps)")):
        v_pred = model(x, t.unsqueeze(0).to(device), ctx)
        x = scheduler.step(v_pred, t, x)

        if i % 5 == 0:
            decoded = vae.decode(x).cpu()
            denoising_frames.append(decoded)

    final_decoded = vae.decode(x).cpu()
    denoising_frames.append(final_decoded)

print(f"Collected {len(denoising_frames)} snapshots.")
print(f"Final video shape: {final_decoded.shape}")

In [ ]:
# Visualize denoising: first temporal frame
fig = visualize_denoising_process(denoising_frames, frame_idx=0, figsize=(18, 3))
plt.show()

# Middle temporal frame
fig = visualize_denoising_process(denoising_frames, frame_idx=8, figsize=(18, 3))
plt.show()

### 7.2 Classifier-Free Guidance (CFG)

CFG improves prompt-following by amplifying the difference between conditioned and unconditioned predictions:

$$\hat{v}_{\text{guided}} = \hat{v}_{\text{uncond}} + w \cdot (\hat{v}_{\text{cond}} - \hat{v}_{\text{uncond}})$$

- $w = 1.0$: Standard conditional generation
- $w > 1.0$: Amplified prompt following (Wan default: $w = 5.0$)

In [ ]:
# Compare CFG scales
cfg_scales = [1.0, 3.0, 5.0]
cfg_results = []
seed = 42

ctx_cond = cached_text(torch.tensor([0], device=device))
ctx_uncond = torch.zeros_like(ctx_cond)

scheduler_cfg = FlowMatchScheduler()

for w in cfg_scales:
    torch.manual_seed(seed)
    x = torch.randn(latent_shape, device=device)
    scheduler_cfg.set_timesteps(30, shift=5.0)

    with torch.no_grad():
        for t in scheduler_cfg.timesteps:
            t_batch = t.unsqueeze(0).to(device)
            v_cond = model(x, t_batch, ctx_cond)
            v_uncond = model(x, t_batch, ctx_uncond)
            v_guided = v_uncond + w * (v_cond - v_uncond)
            x = scheduler_cfg.step(v_guided, t, x)
        video = vae.decode(x).cpu()
    cfg_results.append(video)

fig, axes = plt.subplots(1, len(cfg_scales), figsize=(4 * len(cfg_scales), 4))
for i, (w, video) in enumerate(zip(cfg_scales, cfg_results)):
    frame = video[0, :, 0]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'CFG scale = {w}')
    axes[i].axis('off')

fig.suptitle('Effect of Classifier-Free Guidance Scale', fontsize=14)
plt.tight_layout()
plt.show()

### 7.3 Effect of Step Count

In [ ]:
step_counts = [10, 20, 50]
step_results = []
seed = 123
ctx = cached_text(torch.tensor([0], device=device))

for num_steps in step_counts:
    torch.manual_seed(seed)
    x = torch.randn(latent_shape, device=device)

    sched = FlowMatchScheduler()
    sched.set_timesteps(num_steps, shift=5.0)

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)
        video = vae.decode(x).cpu()
    step_results.append(video)
    print(f"  {num_steps:2d} steps done.")

fig, axes = plt.subplots(2, len(step_counts), figsize=(4 * len(step_counts), 7))
for col, (n, video) in enumerate(zip(step_counts, step_results)):
    for row, (frame_idx, frame_label) in enumerate([(0, 'First frame'), (-1, 'Last frame')]):
        frame = video[0, :, frame_idx]
        frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row, col].imshow(frame)
        axes[row, col].set_title(f'{n} steps - {frame_label}')
        axes[row, col].axis('off')

fig.suptitle('Effect of Denoising Step Count', fontsize=14)
plt.tight_layout()
plt.show()

### 7.4 Save Generated Videos as GIF

In [ ]:
# Generate videos for multiple prompts and save as GIF
num_to_generate = min(num_prompts, 4)
generated_videos = []

sched = FlowMatchScheduler()
sched.set_timesteps(30, shift=5.0)

for prompt_idx in range(num_to_generate):
    torch.manual_seed(prompt_idx * 7 + 42)
    x = torch.randn(latent_shape, device=device)
    ctx = cached_text(torch.tensor([prompt_idx], device=device))

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)
        video = vae.decode(x).cpu()

    generated_videos.append(video[0])
    print(f"  Generated video for prompt {prompt_idx}")

# Save as GIF grid
gif_path = os.path.join(output_dir, 'generated_videos.gif')
save_video_grid(generated_videos, gif_path, nrow=2, fps=8)
print(f"\nSaved video grid to: {gif_path}")

In [ ]:
# Display middle frame from each generated video
fig, axes = plt.subplots(1, num_to_generate, figsize=(4 * num_to_generate, 4))
if num_to_generate == 1:
    axes = [axes]

for i, video in enumerate(generated_videos):
    mid = video.shape[1] // 2
    frame = video[:, mid]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'Prompt {i} (frame {mid})')
    axes[i].axis('off')

fig.suptitle('Generated Videos -- Middle Frame', fontsize=14)
plt.tight_layout()
plt.show()

print("\nDone! You have trained a miniature video diffusion model and generated videos.")
print("The architecture is identical to Wan 2.1 -- only the scale differs.")

---
## Summary

In this notebook you built and trained a complete video diffusion model from scratch:

| What we covered | Key concept |
|----------------|-------------|
| **VAE** | Wan 2.1 VAE compresses 128x128x17 video into [16, 5, 16, 16] latent |
| **Building Blocks** | RoPE, RMSNorm, AdaIN, self/cross-attention, FFN, gating |
| **DiT Architecture** | Same structure as Wan 14B, just smaller (128 vs 5120 dim) |
| **Flow Matching** | Linear interpolation forward, Euler step backward, velocity target |
| **Training** | ~50 epochs on synthetic data with real Wan 2.1 VAE + T5 text embeddings |
| **Inference** | Denoising loop, CFG, step count tradeoffs, WanVAE decode |

### To scale this to production (Wan 2.1):

| Component | Nano | Production |
|-----------|------|------------|
| Model dim | 128 | 5120 |
| Layers | 2 | 40 |
| DiT Parameters | ~2.0M | ~14B |
| VAE | Wan 2.1 (same!) | Wan 2.1 |
| Text encoder | T5 umt5-xxl (same!) | T5 umt5-xxl |
| Data | 60 synthetic clips | Millions of real videos |
| Compute | Minutes on 1 GPU | Weeks on 100s of GPUs |